# Parte 1: Nitratos y Clima
## Riesgo de Contaminación por Nitratos | La Rioja, 2015 - 2025

In [ ]:
# !conda install -c conda-forge geopandas tqdm
# !pip install pandas geopandas openpyxl tqdm

In [1]:
import os, time, warnings
import pandas as pd
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [2]:
SALIDA    = Path(r"C:\Users\mcangulo\OneDrive - FEDERACION DE EMPRESAS DE LA RIOJA\Escritorio\dataset_larioja")
CHE_DIR   = SALIDA / '1_nitratos_che'
AEMET_DIR = SALIDA / '2_clima_aemet'
for d in [CHE_DIR, AEMET_DIR, SALIDA / '9_dataset_final']:
    d.mkdir(parents=True, exist_ok=True)

# Bbox La Rioja (EPSG:4326)
LAT_MIN, LAT_MAX = 41.90, 42.62
LON_MIN, LON_MAX = -3.05, -1.62
# Bbox La Rioja (EPSG:25830) — para filtrar columnas UTM
UTM_X_MIN, UTM_X_MAX = 480_000, 640_000
UTM_Y_MIN, UTM_Y_MAX = 4_630_000, 4_730_000

NO3_UMBRAL_AFECTADA = 37.5   # mg/L — RD 47/2022
NO3_UMBRAL_RIESGO   = 25.0   # mg/L

print('Configuración OK:', SALIDA.exists())

Configuración OK: True


### 1. Construcción de la variable Nº1: Nitratos

Los datos de concentración de nitratos se obtuvieron mediante descarga manual desde el portal de la Confederación Hidrográfica del Ebro (CHE). 

In [3]:
def cargar_archivo(ruta):
    ruta = Path(ruta)
    if not ruta.exists(): raise FileNotFoundError(ruta)
    if ruta.suffix == '.csv':
        return pd.read_csv(ruta, encoding='latin-1', sep=None, engine='python', decimal=',')
    return pd.read_excel(ruta)

df_rnit = cargar_archivo(CHE_DIR / 'nitratos_rnit_raw.xlsx')
df_rbas = cargar_archivo(CHE_DIR / 'nitratos_rbas_raw.xlsx')
print('RNIT:', df_rnit.shape, '| RBAS:', df_rbas.shape)
print('Columnas:', df_rnit.columns.tolist())

RNIT: (7527, 13) | RBAS: (2882, 13)
Columnas: ['IPA', 'TOPONIMIA', 'FECHA DE MUESTREO', 'UTM_X_H30_ETRS89', 'UTM_Y_H30_ETRS89', 'COTA', 'CODIGO MASA DE AGUA', 'MASA DE AGUA', 'NATURALEZA', 'LOCALIDAD', 'MUNICIPIO', 'PROVINCIA', 'Nitratos (mg/L NO3)']


In [4]:
def limpiar_columnas(df):
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(' ','_',regex=False).str.replace('(','',regex=False)
                  .str.replace(')','',regex=False).str.replace('/','_',regex=False)
                  .str.replace('á','a').str.replace('é','e')
                  .str.replace('í','i').str.replace('ó','o').str.replace('ú','u'))
    return df.rename(columns={
        'ipa':'ipa','toponimia':'nombre_punto','fecha_de_muestreo':'fecha',
        'utm_x_h30_etrs89':'x_utm30','utm_y_h30_etrs89':'y_utm30','cota':'cota',
        'codigo_masa_de_agua':'codigo_masa_agua','masa_de_agua':'masa_agua',
        'naturaleza':'naturaleza','localidad':'localidad','municipio':'municipio',
        'provincia':'provincia','nitratos_mg_l_no3':'no3_mgl'})

def limpiar_no3(df):
    if 'no3_mgl' not in df.columns:
        col = next((c for c in df.columns if 'no3' in c or 'nitrat' in c or 'valor' in c), None)
        if col: df = df.rename(columns={col:'no3_mgl'})
    df['no3_mgl'] = (pd.to_numeric(df['no3_mgl'].astype(str)
                     .str.replace(',','.').str.replace('<','').str.strip(), errors='coerce'))
    return df

df_rnit = limpiar_no3(limpiar_columnas(df_rnit)); df_rnit['red'] = 'RNIT'
df_rbas = limpiar_no3(limpiar_columnas(df_rbas)); df_rbas['red'] = 'RBAS'
df      = pd.concat([df_rnit, df_rbas], ignore_index=True)
print('Combinado:', df.shape)
print(df['red'].value_counts())

Combinado: (10409, 14)
red
RNIT    7527
RBAS    2882
Name: count, dtype: int64


In [6]:
# Convertir fecha
df['fecha'] = pd.to_datetime(df['fecha'].astype(str).str.strip(), dayfirst=True, errors='coerce')
df['anio']  = df['fecha'].dt.year
df['mes']   = df['fecha'].dt.month

n_antes = len(df)
df = df.dropna(subset=['no3_mgl','fecha'])
print(f'Eliminadas sin NO3/fecha: {n_antes - len(df)}')

# Filtrar periodo
df = df[df['anio'].between(2015,2025)].copy()

# Eliminar fechas futuras
hoy = pd.Timestamp.today().normalize()
n_ant = len(df)
df = df[df['fecha'] <= hoy].copy()
if n_ant > len(df): print(f'Fechas futuras eliminadas: {n_ant-len(df)}')

# Filtrar a La Rioja
mask_prov = df['provincia'].str.upper().str.contains('RIOJA', na=False) if 'provincia' in df.columns else pd.Series(False,index=df.index)
mask_utm  = df['x_utm30'].between(UTM_X_MIN,UTM_X_MAX) & df['y_utm30'].between(UTM_Y_MIN,UTM_Y_MAX)
mask_lr   = mask_prov & mask_utm if mask_prov.sum()>0 else mask_utm
df = df[mask_lr].copy()

# Duplicados
df = df.drop_duplicates(subset=[c for c in ['ipa','fecha','no3_mgl','red'] if c in df.columns], keep='first')

# Target
df['afectada']  = df['no3_mgl'] > NO3_UMBRAL_AFECTADA
df['en_riesgo'] = df['no3_mgl'].between(NO3_UMBRAL_RIESGO, NO3_UMBRAL_AFECTADA)
df['clase'] = 'normal'
df.loc[df['en_riesgo'],'clase'] = 'riesgo'
df.loc[df['afectada'], 'clase'] = 'afectada'

print(f'Nitratos La Rioja: {len(df):,} filas | {df.ipa.nunique()} puntos únicos')
print(df['clase'].value_counts())
df.to_csv(CHE_DIR/'nitratos_larioja_2015_2025.csv', index=False)

Eliminadas sin NO3/fecha: 0
Nitratos La Rioja: 1,300 filas | 104 puntos únicos
clase
afectada    616
normal      463
riesgo      221
Name: count, dtype: int64


### 2. Construcción de la variable Nº2: Clima

Los datos climáticos diarios se obtienen a través de la API gratuita de AEMET OpenData. 

In [7]:
clima = pd.read_csv(AEMET_DIR / 'aemet_larioja_diario.csv')
print('Clima cargado:', clima.shape)

# Numéricos con coma decimal
for col in ['prec','tmed','tmin','tmax']:
    if col in clima.columns:
        clima[col] = pd.to_numeric(clima[col].astype(str).str.replace(',','.'), errors='coerce')

# Rango físico La Rioja
if 'prec' in clima.columns:  clima.loc[clima.prec<0,'prec'] = 0.0
if 'tmed' in clima.columns:  clima.loc[~clima.tmed.between(-25,45),'tmed'] = float('nan')

clima['fecha'] = pd.to_datetime(clima['fecha'], errors='coerce')
clima['anio']  = clima['fecha'].dt.year
clima['mes']   = clima['fecha'].dt.month
clima = clima[clima['anio'].between(2015,2025)].copy()
clima = clima.drop_duplicates(subset=['indicativo','fecha'], keep='first')
clima = clima.sort_values(['indicativo','fecha']).reset_index(drop=True)

# Interpolar temperatura POR ESTACIÓN (no global)
clima['tmed'] = (clima.groupby('indicativo')['tmed']
                      .transform(lambda x: x.interpolate(method='linear',limit=5,limit_direction='both')))
clima['prec'] = clima['prec'].fillna(0.0)
print(f'Clima limpio: {clima.shape} | Estaciones: {clima.indicativo.nunique()}')

Clima cargado: (30326, 25)
Clima limpio: (29680, 27) | Estaciones: 8


In [8]:
# Variables derivadas con shift(1) — SOLO información PREVIA al muestreo
clima = clima.sort_values(['indicativo','fecha']).reset_index(drop=True)
gp = clima.groupby('indicativo')['prec']
gt = clima.groupby('indicativo')['tmed']

clima['precip_7d']      = gp.transform(lambda x: x.shift(1).rolling(7,  min_periods=3).sum())
clima['precip_15d']     = gp.transform(lambda x: x.shift(1).rolling(15, min_periods=7).sum())
clima['precip_30d']     = gp.transform(lambda x: x.shift(1).rolling(30, min_periods=15).sum())
clima['temp_media_7d']  = gt.transform(lambda x: x.shift(1).rolling(7,  min_periods=3).mean())
clima['temp_media_30d'] = gt.transform(lambda x: x.shift(1).rolling(30, min_periods=15).mean())

media_mes = clima.groupby(['indicativo', clima['fecha'].dt.month])['tmed'].transform('mean')
clima['anomalia_temp'] = clima['tmed'] - media_mes
p90 = clima.groupby('indicativo')['precip_7d'].transform(lambda x: x.quantile(0.90))
clima['evento_extremo_lluvia'] = clima['precip_7d'] > p90

# Añadir coordenadas de estaciones
est_desc = pd.read_excel(AEMET_DIR / 'descripcion_estaciones_larioja.xlsx')
est_desc.columns = (est_desc.columns.str.strip().str.lower()
                    .str.replace('á','a').str.replace('é','e')
                    .str.replace('ó','o').str.replace('ú','u'))
clima['nombre_upper'] = clima['nombre'].astype(str).str.strip().str.upper()
est_desc['nombre_upper'] = est_desc['nombre_estacion'].astype(str).str.strip().str.upper()
clima = clima.merge(est_desc[['nombre_upper','latitud','longitud']], on='nombre_upper', how='left').drop(columns='nombre_upper')

cols_base = ['fecha','indicativo','nombre','altitud','prec','tmed','anio','mes',
             'precip_7d','precip_15d','precip_30d','temp_media_7d','temp_media_30d',
             'anomalia_temp','evento_extremo_lluvia','latitud','longitud']
clima = clima[[c for c in cols_base if c in clima.columns]]
clima.to_csv(AEMET_DIR / 'clima_larioja_2015_2025.csv', index=False)
print('Clima guardado:', clima.shape)

Clima guardado: (29680, 17)


### 3. Integración de las variables 

Antes de poder analizar nitratos y clima juntos, cada punto de muestreo necesita su propia serie climática asociada. Esto se hace en dos pasos:

* Asignación de estación: se usa `sjoin_nearest` para asignar a cada punto RNIT la estación meteorológica más cercana.
  
* Unión por fecha: se usa `merge_asof(direction='backward')` para vincular cada muestra de nitratos con el dato climático más reciente disponible hasta esa fecha, nunca con clima posterior al muestreo, precisamente para no usar información del futuro al construir el dataset.

In [9]:
df_nitratos = pd.read_csv(CHE_DIR / 'nitratos_larioja_2015_2025.csv')
clima       = pd.read_csv(AEMET_DIR / 'clima_larioja_2015_2025.csv')
df_nitratos['fecha'] = pd.to_datetime(df_nitratos['fecha'], errors='coerce')
clima['fecha']       = pd.to_datetime(clima['fecha'],       errors='coerce')

# GeoDataFrames en EPSG:25830
gdf_nit = gpd.GeoDataFrame(df_nitratos.dropna(subset=['x_utm30','y_utm30']),
    geometry=gpd.points_from_xy(df_nitratos.dropna(subset=['x_utm30','y_utm30'])['x_utm30'],
                                df_nitratos.dropna(subset=['x_utm30','y_utm30'])['y_utm30']), crs='EPSG:25830')
est = clima[['indicativo','nombre','latitud','longitud']].drop_duplicates().dropna()
gdf_est = gpd.GeoDataFrame(est, geometry=gpd.points_from_xy(est.longitud,est.latitud), crs='EPSG:4326').to_crs('EPSG:25830')

# Join espacial: estación más cercana
gdf_j = gpd.sjoin_nearest(gdf_nit, gdf_est[['indicativo','nombre','geometry']],
                           how='left', distance_col='dist_estacion_m')
gdf_j = gdf_j.rename(columns={'indicativo_right':'indicativo_estacion','nombre_right':'nombre_estacion'}).drop(columns=['index_right'], errors='ignore')
print('sjoin OK:', gdf_j.shape)
print(f"Puntos >20km de estacion: {(gdf_j.dist_estacion_m>20000).sum()} ({100*(gdf_j.dist_estacion_m>20000).mean():.1f}%)")

sjoin OK: (1300, 23)
Puntos >20km de estacion: 202 (15.5%)


**Interpretación:**

* El join espacial funcionó, pero hay una parte importante de muestras, el 15.5%, cuya estación climática asignada está relativamente lejos.

* Eso no significa necesariamente que esté mal, pero sí es una advertencia: para esos 202 puntos, las variables climáticas podrían representar peor las condiciones locales del punto de nitratos.

* Para un análisis ambiental, podrías considerar marcar esos puntos como de menor confianza, revisar su distribución en un mapa, o probar umbrales como 10 km, 20 km y 30 km para ver cómo cambia la cobertura.

In [11]:
print(df_j.columns.tolist())

['ipa', 'nombre_punto', 'fecha', 'x_utm30', 'y_utm30', 'cota', 'codigo_masa_agua', 'masa_agua', 'naturaleza', 'localidad', 'municipio', 'provincia', 'no3_mgl', 'red', 'anio', 'mes', 'afectada', 'en_riesgo', 'clase', 'indicativo', 'nombre', 'dist_estacion_m']


In [12]:
df_j = pd.DataFrame(gdf_j.drop(columns='geometry')).copy()
df_j['indicativo']  = df_j['indicativo'].astype(str)
clima['indicativo']          = clima['indicativo'].astype(str)
df_j  = df_j.sort_values(['indicativo','fecha'])
clima = clima.sort_values(['indicativo','fecha'])

resultados = []
for est in df_j['indicativo'].unique():
    left  = df_j[df_j['indicativo']==est].copy()
    right = clima[clima['indicativo']==est].copy()
    if right.empty: resultados.append(left); continue
    merged = pd.merge_asof(left.sort_values('fecha'), right.sort_values('fecha'),
                           on='fecha', direction='backward',  
                           tolerance=pd.Timedelta('7D'), suffixes=('','_clima'))
    resultados.append(merged)

df_modelo = pd.concat(resultados, ignore_index=True)

# Limpiar columnas duplicadas
cols_drop = [c for c in ['index_right','provincia_y','municipio_y','anio_y','mes_y'] if c in df_modelo.columns]
df_modelo = df_modelo.drop(columns=cols_drop)
df_modelo.columns = [c.replace('_x','') for c in df_modelo.columns]

# Eliminar filas sin clima
cols_clim = [c for c in ['prec','tmed','precip_7d'] if c in df_modelo.columns]
n_antes = len(df_modelo)
df_modelo = df_modelo.dropna(subset=cols_clim).copy()
print(f'Eliminadas sin clima: {n_antes-len(df_modelo)}')
print(f'Dataset final P1: {df_modelo.shape}')

Eliminadas sin clima: 127
Dataset final P1: (1173, 38)


**Interpretación:**

Eso significa que de las 1300 muestras iniciales, 127 no encontraron datos climáticos válidos dentro del criterio temporal usado.

El dataset final queda con:

* 1173 filas: muestras de nitratos que sí tienen estación asignada y datos climáticos válidos.

* 38 columnas: variables originales de nitratos más variables de estación y clima.

In [13]:
# Separación explícita TARGET / FEATURES — evitar leakage
TARGET = 'clase'
EXCLUIR_FEATURES = {'no3_mgl','afectada','en_riesgo','clase',
                    'ipa','fecha','nombre_punto','municipio','provincia',
                    'masa_agua','codigo_masa_agua','red',
                    'indicativo_estacion','nombre_estacion'}
FEATURES_P1 = [c for c in df_modelo.columns
               if c not in EXCLUIR_FEATURES and df_modelo[c].dtype in ['float64','int64','bool']]

print('Target:', TARGET)
print(f'Features disponibles P1 ({len(FEATURES_P1)}):', FEATURES_P1)
print('\nDistribución target:')
print(df_modelo[TARGET].value_counts())
print(df_modelo[TARGET].value_counts(normalize=True).mul(100).round(1))
print('\nAVISO: clase riesgo ~14% — usar class_weight=balanced')

Target: clase
Features disponibles P1 (19): ['x_utm30', 'y_utm30', 'cota', 'anio', 'mes', 'dist_estacion_m', 'altitud', 'prec', 'tmed', 'anio_clima', 'mes_clima', 'precip_7d', 'precip_15d', 'precip_30d', 'temp_media_7d', 'temp_media_30d', 'anomalia_temp', 'latitud', 'longitud']

Distribución target:
clase
afectada    591
normal      410
riesgo      172
Name: count, dtype: int64
clase
afectada    50.4
normal      35.0
riesgo      14.7
Name: proportion, dtype: float64

AVISO: clase riesgo ~14% — usar class_weight=balanced


**Observación:**

La clase "riesgo" representa solo el 14,7% de las observaciones, frente al 50,4% de "afectada" y el 35% de "normal". Este desbalance se aborda explícitamente en la Parte 8 (Modelado), donde se evalúa con F1-score y recall de la clase "riesgo" como métricas principales, no con accuracy global, y se comparan estrategias como `class_weight='balanced'` y SMOTE. En este proyecto, un falso negativo (no detectar una zona realmente en riesgo) tiene consecuencias ambientales reales, por lo que minimizarlo es prioritario frente a maximizar el accuracy general.

In [14]:
print('=== VERIFICACIÓN FINAL P1 ===')
print(f'Filas:    {len(df_modelo):,}')
print(f'Columnas: {df_modelo.shape[1]}')
print(f'Puntos:   {df_modelo.ipa.nunique()}')
print(f'Rango:    {df_modelo.fecha.min().date()} → {df_modelo.fecha.max().date()}')
print(f'NO3 media:{df_modelo.no3_mgl.mean():.1f} mg/L')
nulos = df_modelo[FEATURES_P1+[TARGET]].isna().sum()
print('Nulos features/target:', nulos[nulos>0].to_dict() or 'ninguno')

ruta = SALIDA / '9_dataset_final' / 'dataset_nitratos_clima_larioja_final.csv'
df_modelo.to_csv(ruta, index=False)
print('Guardado:', ruta)

=== VERIFICACIÓN FINAL P1 ===
Filas:    1,173
Columnas: 38
Puntos:   101
Rango:    2015-03-10 → 2025-12-17
NO3 media:51.8 mg/L
Nulos features/target: {'cota': 426, 'precip_30d': 1, 'temp_media_30d': 1}
Guardado: C:\Users\mcangulo\OneDrive - FEDERACION DE EMPRESAS DE LA RIOJA\Escritorio\dataset_larioja\9_dataset_final\dataset_nitratos_clima_larioja_final.csv
